In [ ]:
# Run first: locate the repository root from any notebook subfolder.
from pathlib import Path
import os
import sys
project_root = next((candidate for candidate in [Path.cwd(), *Path.cwd().parents]
                     if (candidate / "app.py").is_file()
                     and (candidate / "retrieval_final.py").is_file()), None)
if project_root is None:
    raise RuntimeError("Start Jupyter within the project repository.")
os.chdir(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))


# Full-corpus E5 dense retrieval evaluation

In [ ]:
# Configure the E5 model, FAISS index, evaluation files, and retrieval settings.
import json
import time
from pathlib import Path

import faiss
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from IPython.display import display
from sentence_transformers import SentenceTransformer

EMAILS_FILE = Path('enron_clean.csv')
EVALUATION_FILE = Path('data/enron_evaluation_250.csv')
EMBEDDING_DIRECTORY = Path('e5_embeddings_final')
MANIFEST_FILE = EMBEDDING_DIRECTORY / 'embedding_manifest.json'
DENSE_INDEX_FILE = EMBEDDING_DIRECTORY / 'e5_dense_index.faiss'
MODEL_NAME = 'intfloat/e5-base-v2'
QUERY_PREFIX = 'query: '
EMAIL_RESULTS_NEEDED = 100
INITIAL_CHUNK_FETCH = 5_000
TOP_K_VALUES = [1, 3, 5, 10, 50, 100]

In [ ]:
# Load and validate the corpus, evaluation set, embedding manifest, and FAISS index.
emails = pd.read_csv(EMAILS_FILE)
evaluation_questions = pd.read_csv(EVALUATION_FILE)
manifest = json.loads(MANIFEST_FILE.read_text(encoding='utf-8'))
if not DENSE_INDEX_FILE.exists():
    raise FileNotFoundError('Run the dense-index cell at the end of Embeddings-Final.ipynb first.')
dense_index = faiss.read_index(str(DENSE_INDEX_FILE))

required_email_columns = {'email_id', 'sender', 'recipients', 'subject', 'body'}
required_question_columns = {'question', 'answer', 'email_id', 'difficulty', 'retrieval_type'}
missing_email_columns = required_email_columns - set(emails.columns)
missing_question_columns = required_question_columns - set(evaluation_questions.columns)
if missing_email_columns or missing_question_columns:
    raise ValueError(
        f'Missing email columns: {sorted(missing_email_columns)}; '
        f'missing question columns: {sorted(missing_question_columns)}'
    )
if not manifest.get('complete') or manifest['model_name'] != MODEL_NAME:
    raise ValueError('The completed embedding manifest does not match this E5 evaluation.')
if dense_index.ntotal != manifest['number_of_chunks'] or dense_index.d != manifest['embedding_dimensions']:
    raise ValueError('The FAISS index does not match the completed embedding manifest.')

emails['email_id'] = emails['email_id'].astype(str)
evaluation_questions['email_id'] = evaluation_questions['email_id'].astype(str)
email_ids = emails['email_id'].to_numpy()
email_position_by_id = {email_id: position for position, email_id in enumerate(email_ids)}
missing_sources = set(evaluation_questions['email_id']) - set(email_position_by_id)
if missing_sources:
    raise ValueError(f'{len(missing_sources)} labelled source emails are missing.')
target_positions = np.asarray(
    [email_position_by_id[email_id] for email_id in evaluation_questions['email_id']],
    dtype=np.int64,
)
print(f'Emails: {len(emails):,}')
print(f'Questions: {len(evaluation_questions):,}')
print(f'FAISS vectors: {dense_index.ntotal:,}')
print(f"Chunk size / overlap: {manifest['chunk_size']} / {manifest['chunk_overlap']}")

In [ ]:
# Confirm the existing easy, medium, and hard evaluation groups before retrieval.
difficulty_counts = (
    evaluation_questions['difficulty'].value_counts()
    .reindex(['easy', 'medium', 'hard'])
    .rename_axis('difficulty')
    .reset_index(name='questions')
)
display(difficulty_counts)

In [ ]:
# Map each FAISS vector row back to the email that produced the corresponding chunk.# This reads only source_row, not the chunk text or embedding arrays.
chunk_source_rows = np.empty(dense_index.ntotal, dtype=np.int32)
offset = 0
for shard in manifest['shards']:
    source_rows = pd.read_parquet(
        Path(shard['chunks_file']), columns=['source_row']
    )['source_row'].to_numpy(dtype=np.int32)
    next_offset = offset + len(source_rows)
    chunk_source_rows[offset:next_offset] = source_rows
    offset = next_offset
if offset != dense_index.ntotal:
    raise ValueError('Chunk lookup length does not match the FAISS index.')
print(f'Created source lookup for {len(chunk_source_rows):,} vector rows.')

In [ ]:
# Encode the evaluation questions using the E5 query prefix and normalized embeddings.
if torch.backends.mps.is_available():
    device = 'mps'
else:
    device = 'cpu'
embedding_model = SentenceTransformer(MODEL_NAME, device=device)
question_embeddings = embedding_model.encode(
    [QUERY_PREFIX + str(question) for question in evaluation_questions['question']],
    batch_size=64, convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=True,
).astype(np.float32)
print(f'Embedded {len(question_embeddings):,} questions on {device}.')

In [ ]:
# Search the FAISS index and convert chunk results into unique email-level rankings.
search_started = time.perf_counter()
chunk_fetch = min(INITIAL_CHUNK_FETCH, dense_index.ntotal)
while True:
    chunk_scores, chunk_vector_ids = dense_index.search(question_embeddings, chunk_fetch)
    email_rankings = []
    for vector_ids in chunk_vector_ids:
        seen = set()
        ranking = []
        for vector_id in vector_ids:
            email_position = int(chunk_source_rows[vector_id])
            if email_position not in seen:
                seen.add(email_position)
                ranking.append(email_position)
                if len(ranking) == EMAIL_RESULTS_NEEDED:
                    break
        email_rankings.append(np.asarray(ranking, dtype=np.int64))
    if min(map(len, email_rankings)) >= EMAIL_RESULTS_NEEDED or chunk_fetch == dense_index.ntotal:
        break
    chunk_fetch = min(chunk_fetch * 2, dense_index.ntotal)

relevant_ranks = []
for ranking, target_position in zip(email_rankings, target_positions):
    matches = np.flatnonzero(ranking == target_position)
    relevant_ranks.append(int(matches[0]) + 1 if len(matches) else np.inf)
relevant_ranks = np.asarray(relevant_ranks, dtype=float)
evaluation_questions['e5_rank'] = relevant_ranks
print(f'Searched {chunk_fetch:,} chunks per question.')
print(f'Index evaluation finished in {time.perf_counter() - search_started:.1f} seconds.')

In [ ]:
# Calculate overall and difficulty-level Recall@K and MRR for E5 retrieval.
def metric_row(label, ranks, label_column='method'):
    row = {label_column: label, 'questions': len(ranks)}
    for k in TOP_K_VALUES:
        row[f'recall_at_{k}'] = float(np.mean(ranks <= k))
    row['mrr_at_100'] = float(np.mean(1.0 / ranks))
    return row

overall_results = pd.DataFrame([metric_row('Dense E5 FAISS', relevant_ranks)])
difficulty_results = pd.DataFrame([
    metric_row(difficulty, group['e5_rank'].to_numpy(dtype=float), 'difficulty')
    for difficulty, group in evaluation_questions.groupby('difficulty', sort=False)
])
formatters = {**{f'recall_at_{k}': '{:.1%}' for k in TOP_K_VALUES}, 'mrr_at_100': '{:.3f}'}
display(overall_results.style.format(formatters))
display(difficulty_results.style.format(formatters).background_gradient(
    subset=[f'recall_at_{k}' for k in TOP_K_VALUES] + ['mrr_at_100'], cmap='YlGn'
))

In [ ]:
# Plot Recall@10 for the easy, medium, and hard question groups.
plt.figure(figsize=(11, 5))
sns.barplot(data=difficulty_results, x='difficulty', y='recall_at_10', order=['easy', 'medium', 'hard'], color='#59A14F')
plt.ylim(0, 1)
plt.xlabel('Question difficulty')
plt.ylabel('Recall@10')
plt.title('Full-corpus E5 Recall@10 by difficulty')
plt.tight_layout()
plt.show()